Show examples of each of these cases of integration, with a good example and wrong example.
- Model: seamless
- Evaluation set: AI papers & model cards
- Language: all the five languages
- Good and bad examples according to: COMET scores

As a result, we will have a table of 2 columns (good, bad) and 3x5 rows (prompting, word alignment, word alignment+prompting)x(languages, or nx5 rows if we want to show more than one example).

In [1]:
import json
import numpy as np
import pandas as pd
tgt_langs = [
    "Arabic",
    "Chinese",
    "French",
    "Japanese",
    "Russian",
]

models = {
    'nllb': 'nllb',
    'seamless': 'seamless',
    'gpt4omini': 'gpt-4o-mini',
    'aya_old': 'aya-23-8B',
    'aya': 'aya-expanse-8B'
}

eval_data = [json.loads(i) for i in open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval_data_google.jsonl", 'r').readlines()]

In [2]:
methods = {
    '_hard_replace': 'Word Alignment',
    '_prompt_gpt4omini': 'Prompting Refinement'
}

examples = {}

for model, model_name in models.items():
    
    examples_model = {}
    
    for method, method_name in methods.items():
        # load data from a method
        method_data = [json.loads(i) for i in open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/output/{model}{method}.jsonl", 'r').readlines()]
        
        method_data_res = json.load(open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval/{model}{method}_comet.json", 'r'))
        
        examples_method = {}
        
        for lang in tgt_langs:
            method_data_res[lang] = np.array(method_data_res[f'{lang}_scores'])
            diff = method_data_res[lang]
            
            # get index
            # Number of maximum and minimum values to retrieve
            n = 5

            # get good example
            # Get the indices of the top n maximum values
            max_indices = np.argsort(diff)[-n:][::-1]  # Sort and take the last n indices, then reverse
            max_values = diff[max_indices]  # Use these indices to get the values
            # print(max_values)
            for idx, index in enumerate(max_indices):
                good = {}
                good[f'COMET Score'] = max_values[idx]
                good[f'English'] = eval_data[index]['text'] # English text
                good[f'{lang} Ground Truth'] = eval_data[index][f'text_{lang}'] # ground truth result
                good[f'{lang} {method_name}'] = method_data[index][f'text_{lang}'] # method translation result
                
                examples_method[f'{lang} Good {idx+1}'] = good

            # get bad example
            # Get the indices of the top n minimum values
            min_indices = np.argsort(diff)[:n]  # Sort and take the first n indices
            min_values = diff[min_indices]  # Use these indices to get the values
            
            for idx, index in enumerate(min_indices):
                bad = {}
                bad[f'COMET Score'] = min_values[idx]
                bad[f'English'] = eval_data[index]['text'] # English text
                bad[f'{lang} Ground Truth'] = eval_data[index][f'text_{lang}'] # ground truth result
                bad[f'{lang} {method_name}'] = method_data[index][f'text_{lang}'] # method translation result
                
                examples_method[f'{lang} Bad {idx+1}'] = bad
            
            
        examples_model[method_name] = examples_method
        
    examples[model_name] = examples_model

In [3]:
import pandas as pd

# Create an empty dictionary to store the dataframes
dataframes = {}

# Loop through each model and language
for model_name, model_data in examples.items():
    for lang in tgt_langs:
        # Initialize lists to store data for the dataframe
        method_list = []
        example_type_list = []  # Good or Bad
        example_score_list = []
        
        tmp_list = []
        
        english_text_list = []
        ground_truth_list = []
        direct_translation_list = []
        method_translation_list = []

        # Iterate through methods
        for method_name, method_data in model_data.items():
            # Iterate through Good and Bad examples
            for example_type in ['Good', 'Bad']:
                for idx in range(1, 6):  # n examples for each (Good and Bad)
                    key = f'{lang} {example_type} {idx}'
                    if key in method_data:
                        example = method_data[key]
                        
                        
                        # method_list.append(method_name)
                        # example_type_list.append(example_type)
                        # example_score_list.append(example['COMET Score'])
                        

                        # tmp_list.append("\multicolumn{2}{l}{\\textit{\\textbf{" + f"Method: {method_name}; Example: {example_type}, COMET Score: {example['COMET Score']:.4f}" + "}" + "}" + "}")
                        
                        tmp_list.append(f"Method: {method_name}; Example: {example_type}, COMET Score: {example['COMET Score']:.4f}")
                        
                        english_text_list.append(example['English'].replace("&", "\&"))
                        
                        ground_truth_list.append(example[f'{lang} Ground Truth'].replace("&", "\&"))
                        
                        method_translation_list.append(example[f'{lang} {method_name}'].replace("&", "\&"))

        # Create a dataframe for the current model and language
        df = pd.DataFrame({
            # 'Method': method_list,
            # 'Example Type': example_type_list,
            # 'COMET Score': example_score_list,
            
            'Information': tmp_list,
            'English Text': english_text_list,
            f'Ground Truth': ground_truth_list,
            f'Method Translation': method_translation_list
        })
        

        # Assuming your original dataframe is named df
        # Create an empty list to store rows for the new dataframe
        new_rows = []

        # Loop through each row in the original dataframe
        for i in range(len(df)):
            # Extract data for the current row
            row = df.iloc[i]
            for column in df.columns:
                # Append a new row for each column
                new_rows.append({
                    # 'Key': column,       # Column name becomes the key
                    'Value': column + ": " + row[column] if column != 'Information' else row[column] # Cell value becomes the value 
                })

        # Create the new dataframe
        new_df = pd.DataFrame(new_rows)

        # Reset the index for better readability
        new_df = new_df.reset_index(drop=True)

        # Store the dataframe in the dictionary
        dataframes[f'{model_name}_{lang}'] = new_df
        

# # Display or save the dataframes
# for key, df in dataframes.items():
#     print(f"DataFrame: {key}")
#     print(df)


In [8]:
idx = 4
key, df = list(dataframes.items())[idx]
print(f"DataFrame: {key}")
# print(df.to_latex(index=False))
print("\n".join(df['Value'].tolist()))

DataFrame: nllb_Russian
Method: Word Alignment; Example: Good, COMET Score: 0.9735
English Text: Using this approach, a BRDF can be measured in just a few minutes.
Ground Truth: Используя этот подход, BRDF можно измерить всего за несколько минут.
Method Translation: Используя этот подход, BRDF можно измерить всего за несколько минут.
Method: Word Alignment; Example: Good, COMET Score: 0.9712
English Text: The experiment was implemented in Python 3.6 and the packages listed in table 11
Ground Truth: Эксперимент был реализован на Python 3.6 и пакетах, перечисленных в таблице 11.
Method Translation: Эксперимент был реализован в Python 3.6 и пакетах, перечисленных в таблице 11.
Method: Word Alignment; Example: Good, COMET Score: 0.9632
English Text: I D P K Q N F Y M H W C BLOSUM62 ( reference )
Ground Truth: I D P K Q N F Y M H W C BLOSUM62 (ссылка)
Method Translation: I D P K Q N F Y M H W C BLOSUM62 (ссылка)
Method: Word Alignment; Example: Good, COMET Score: 0.9627
English Text: Theref